# **Feature engineering of News datasets**

In [25]:
import os
import re
import json
import ast
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from dotenv import load_dotenv


In [26]:
load_dotenv()
FEATURES = os.getenv("FEATURES")
DATA_DIR = os.getenv("DATA_DIR")
FEAT_DIR  = os.path.join(FEATURES, "features")
IN_CSV    = os.path.join(DATA_DIR, "preprocessed.csv")
os.makedirs(FEAT_DIR, exist_ok=True)


In [27]:
KAZ_STOPWORDS = list({
    "және", "да", "де", "бар", "жоқ", "бұл", "осы", "оның", "ол",
    "олар", "біз", "сіз", "мен", "сен", "үшін", "деп", "болды",
    "болған", "болып", "деген", "туралы", "екен", "еді", "алды",
    "қол", "кейін", "бойынша", "арқылы", "ретінде", "дейін",
    "жылы", "жыл", "ай", "күн", "қаласы", "қаласында", "облысы",
    "облысында", "республикасы", "оны", "оларды", "барлық", "онда",
    "жасалды", "болса", "сондай", "сонымен", "сол", "соның",
    "бірге", "бірақ", "немесе", "не", "яғни", "осыған", "үстінде",
    "астында", "қарай", "қатысты", "хабарлады", "айтты", "мәлімет",
    "хабарлайды", "деді", "берді", "жасады", "атап", "тыс",
})


# **Loading merged dataset**

In [28]:
df = pd.read_csv(IN_CSV)
df = df[df["text"].notna()].copy()
df = df.reset_index(drop=True)

print(f"Labeled rows : {len(df):,}")
print(f"Categories   : {df['category'].value_counts().to_dict()}")


Labeled rows : 27,743
Categories   : {'other': 18046, 'health': 2854, 'politics': 1953, 'sports': 1242, 'crime': 1152, 'international': 912, 'economy': 642, 'education': 528, 'social': 414}


In [29]:
df.shape

(27743, 10)

In [30]:
df["input_text"] = df["title"].fillna("") + " " + \
                   df["title"].fillna("") + " " + \
                   df["text"].fillna("")

# Light normalisation: lowercase, collapse whitespace
def normalise(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["input_text"] = df["input_text"].apply(normalise)
print("Sample input_text:", df["input_text"].iloc[0][:120], "...")

Sample input_text: тоқаев қазақстандықтарды 2022 жылы не күтіп тұрғанын айтты тоқаев қазақстандықтарды 2022 жылы не күтіп тұрғанын айтты пр ...


# **train/val/test split 10/15/15**

In [31]:
idx = np.arange(len(df))
y   = df["category"].values

idx_train, idx_temp, _, y_temp = train_test_split(
    idx, y, test_size=0.30, stratify=y, random_state=42
)
idx_val, idx_test, _, _ = train_test_split(
    idx_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Train : {len(idx_train):,} rows")
print(f"Val   : {len(idx_val):,} rows")
print(f"Test  : {len(idx_test):,} rows")

np.save(os.path.join(FEAT_DIR, "train_idx.npy"), idx_train)
np.save(os.path.join(FEAT_DIR, "val_idx.npy"),   idx_val)
np.save(os.path.join(FEAT_DIR, "test_idx.npy"),  idx_test)

Train : 19,420 rows
Val   : 4,161 rows
Test  : 4,162 rows


# **Label encoding**

In [32]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["category"])
label_map = {cat: int(idx) for idx, cat in enumerate(le.classes_)}
print("Label map:", label_map)

with open(os.path.join(FEAT_DIR, "label_encoder.json"), "w", encoding="utf-8") as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)

df[["category", "label"]].to_csv(
    os.path.join(FEAT_DIR, "labels.csv"), index=False, encoding="utf-8"
)
print(f"Saved labels.csv and label_encoder.json")


Label map: {'crime': 0, 'economy': 1, 'education': 2, 'health': 3, 'international': 4, 'other': 5, 'politics': 6, 'social': 7, 'sports': 8}
Saved labels.csv and label_encoder.json


# **TF-IDF word features**

In [33]:
texts_train = df["input_text"].iloc[idx_train].values

tfidf_word = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=80_000,
    min_df=3,
    sublinear_tf=True,
    stop_words=KAZ_STOPWORDS,
    token_pattern=r"[а-яёәғіқңөүһa-z]{2,}",  # Cyrillic + Latin, min 2 chars
)
tfidf_word.fit(texts_train)

X_word = tfidf_word.transform(df["input_text"].values)
print(f"Word TF-IDF matrix : {X_word.shape}")

sp.save_npz(os.path.join(FEAT_DIR, "tfidf_word.npz"), X_word)
with open(os.path.join(FEAT_DIR, "tfidf_word_vocab.json"), "w", encoding="utf-8") as f:
    json.dump({k: int(v) for k, v in tfidf_word.vocabulary_.items()}, f, ensure_ascii=False)
print("Saved tfidf_word.npz + vocab")


Word TF-IDF matrix : (27743, 80000)
Saved tfidf_word.npz + vocab


# **TF-IDF character n-gram features**

In [34]:
tfidf_char = TfidfVectorizer(
    analyzer="char_wb",      # char n-grams within word boundaries
    ngram_range=(2, 4),
    max_features=60_000,
    min_df=5,
    sublinear_tf=True,
)
tfidf_char.fit(texts_train)

X_char = tfidf_char.transform(df["input_text"].values)
print(f"Char TF-IDF matrix : {X_char.shape}")

sp.save_npz(os.path.join(FEAT_DIR, "tfidf_char.npz"), X_char)
with open(os.path.join(FEAT_DIR, "tfidf_char_vocab.json"), "w", encoding="utf-8") as f:
    json.dump({k: int(v) for k, v in tfidf_char.vocabulary_.items()}, f, ensure_ascii=False)
print("Saved tfidf_char.npz + vocab")


Char TF-IDF matrix : (27743, 60000)
Saved tfidf_char.npz + vocab


# **Combined TF-IDF (word + char)**

In [35]:
X_combined = sp.hstack([X_word, X_char], format="csr")
print(f"Combined TF-IDF matrix : {X_combined.shape}")
sp.save_npz(os.path.join(FEAT_DIR, "tfidf_combined.npz"), X_combined)
print("Saved tfidf_combined.npz")


Combined TF-IDF matrix : (27743, 140000)
Saved tfidf_combined.npz


# **Hand-crafted statistical features**

In [36]:
def hand_features(row) -> dict:
    text   = str(row["text"])
    title  = str(row["title"])
    words  = text.split()
    n_char = len(text)
    n_word = len(words)

    # Average word length
    avg_word_len = (sum(len(w) for w in words) / n_word) if n_word else 0

    # Unique word ratio (lexical diversity)
    unique_ratio = len(set(w.lower() for w in words)) / n_word if n_word else 0

    # Punctuation ratio
    punct = sum(1 for c in text if c in ".,;:!?–—\"'«»()")
    punct_ratio = punct / n_char if n_char else 0

    # Digit ratio
    digits = sum(1 for c in text if c.isdigit())
    digit_ratio = digits / n_char if n_char else 0

    # Sentence count (rough: split on . ! ?)
    sentences = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
    n_sent = max(len(sentences), 1)

    # Average sentence length in words
    avg_sent_len = n_word / n_sent

    # Title–body overlap: ratio of title words found in body
    title_words = set(title.lower().split())
    body_words  = set(w.lower() for w in words)
    overlap = len(title_words & body_words) / len(title_words) if title_words else 0

    return {
        "char_count":       n_char,
        "word_count":       n_word,
        "title_len":        len(title),
        "avg_word_len":     round(avg_word_len, 3),
        "unique_word_ratio": round(unique_ratio, 4),
        "punct_ratio":      round(punct_ratio, 4),
        "digit_ratio":      round(digit_ratio, 4),
        "sentence_count":   n_sent,
        "avg_sentence_len": round(avg_sent_len, 3),
        "title_in_text":    round(overlap, 4),
        "category":         row["category"],
        "label":            row["label"],
    }

feat_rows = [hand_features(row) for _, row in df.iterrows()]
df_feat   = pd.DataFrame(feat_rows)

In [37]:
print("Hand-crafted feature sample:")
print(df_feat[["char_count", "word_count", "avg_word_len",
               "unique_word_ratio", "sentence_count", "category"]].head(5).to_string(index=False))

df_feat.to_csv(os.path.join(FEAT_DIR, "handcrafted.csv"), index=False, encoding="utf-8")
print(f"\nSaved handcrafted.csv  ({len(df_feat):,} rows × {len(df_feat.columns)} cols)")


Hand-crafted feature sample:
 char_count  word_count  avg_word_len  unique_word_ratio  sentence_count  category
       1378         165         7.358             0.7939              17  politics
       1088         134         7.127             0.6418              11  politics
       3415         385         7.873             0.7221              30  politics
       1205         147         7.204             0.7687              18    health
       1031         124         7.323             0.7500              11 education

Saved handcrafted.csv  (27,743 rows × 12 cols)


In [38]:
for fname in sorted(os.listdir(FEAT_DIR)):
    fpath = os.path.join(FEAT_DIR, fname)
    size  = os.path.getsize(fpath) / 1e6
    print(f"  {fname:35s}  {size:6.1f} MB")

print("""
Usage in next scripts
─────────────────────
  import scipy.sparse as sp, numpy as np, pandas as pd, json

  X_word     = sp.load_npz('data/features/tfidf_word.npz')
  X_char     = sp.load_npz('data/features/tfidf_char.npz')
  X_combined = sp.load_npz('data/features/tfidf_combined.npz')
  df_hand    = pd.read_csv('data/features/handcrafted.csv')
  train_idx  = np.load('data/features/train_idx.npy')
  val_idx    = np.load('data/features/val_idx.npy')
  test_idx   = np.load('data/features/test_idx.npy')
  label_map  = json.load(open('data/features/label_encoder.json'))
""")


  handcrafted.csv                         1.7 MB
  label_encoder.json                      0.0 MB
  labels.csv                              0.2 MB
  test_idx.npy                            0.0 MB
  tfidf_char.npz                        390.4 MB
  tfidf_char_vocab.json                   1.0 MB
  tfidf_combined.npz                    431.1 MB
  tfidf_word.npz                         39.2 MB
  tfidf_word_vocab.json                   2.7 MB
  train_idx.npy                           0.2 MB
  val_idx.npy                             0.0 MB

Usage in next scripts
─────────────────────
  import scipy.sparse as sp, numpy as np, pandas as pd, json

  X_word     = sp.load_npz('data/features/tfidf_word.npz')
  X_char     = sp.load_npz('data/features/tfidf_char.npz')
  X_combined = sp.load_npz('data/features/tfidf_combined.npz')
  df_hand    = pd.read_csv('data/features/handcrafted.csv')
  train_idx  = np.load('data/features/train_idx.npy')
  val_idx    = np.load('data/features/val_idx.npy')
  test_

In [39]:
# Training dataset (Kaggle) — used for train/val/test
df_train = df[df["source"] == "kaggle"].copy()

# Kazakhstan inference dataset (your parsed data)
df_kaz = df[df["source"] == "parsed"].copy()

df_train.to_csv(os.path.join(DATA_DIR, "kaggle_train_dataset.csv"), index=False, encoding="utf-8")
df_kaz.to_csv(os.path.join(DATA_DIR, "kazakhstan_inference_dataset.csv"), index=False, encoding="utf-8")

print(f"Training dataset     : {len(df_train):,} rows → kaggle_train_dataset.csv")
print(f"Kazakhstan inference : {len(df_kaz):,} rows  → kazakhstan_inference_dataset.csv")

Training dataset     : 16,230 rows → kaggle_train_dataset.csv
Kazakhstan inference : 11,513 rows  → kazakhstan_inference_dataset.csv
